# Car Resale Price Prediction: Decision Tree vs Random Forest vs XGBoost
Dataset: [Vehicle dataset from CarDekho](https://www.kaggle.com/datasets/nehalbirla/vehicle-dataset-from-cardekho) (`car data.csv`), prices in lakhs.

**Metrics:** R², Adjusted R², MAE, MSE, RMSE, MAPE, Median AE, Max Error, Explained Variance, RMSLE, 5-fold CV R², train time.

In [ ]:
# Run once if needed:
# !pip install xgboost kagglehub

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, KFold, cross_val_score, RandomizedSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             median_absolute_error, max_error, explained_variance_score,
                             mean_absolute_percentage_error, mean_squared_log_error)

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
RANDOM_STATE = 42

## 1. Load data

In [ ]:
# Option A: local file
df = pd.read_csv("car data.csv")

# Option B: download with kagglehub
# import kagglehub, os
# path = kagglehub.dataset_download("nehalbirla/vehicle-dataset-from-cardekho")
# df = pd.read_csv(os.path.join(path, "car data.csv"))

df.head()

## 2. Explore

In [ ]:
print(df.shape)
df.info()
print("\nMissing:\n", df.isnull().sum())
print("Duplicates:", df.duplicated().sum())
df.describe()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(df["Selling_Price"], kde=True, ax=ax[0])
sns.scatterplot(x="Present_Price", y="Selling_Price", data=df, ax=ax[1])
sns.boxplot(x="Fuel_Type", y="Selling_Price", data=df, ax=ax[2])
plt.tight_layout(); plt.show()

## 3. Preprocessing

In [ ]:
df = df.drop_duplicates().copy()
df["Car_Age"] = pd.Timestamp.today().year - df["Year"]
df = df.drop(columns=["Car_Name", "Year"])
df = pd.get_dummies(df, columns=["Fuel_Type", "Seller_Type", "Transmission"], drop_first=True)

plt.figure(figsize=(9, 6))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation matrix"); plt.show()

X = df.drop(columns=["Selling_Price"])
y = df["Selling_Price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print(X_train.shape, X_test.shape)

## 4. Models and metrics

In [ ]:
models = {
    "Decision Tree": DecisionTreeRegressor(random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost":       XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4,
                                  random_state=RANDOM_STATE, n_jobs=-1, verbosity=0),
}

def adjusted_r2(r2, n, p):
    return 1 - (1 - r2) * (n - 1) / (n - p - 1)

def all_metrics(y_true, y_pred, n_features):
    r2 = r2_score(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmsle = np.sqrt(mean_squared_log_error(y_true, np.clip(y_pred, 0, None)))
    return {
        "R2":           r2,
        "Adj_R2":       adjusted_r2(r2, len(y_true), n_features),
        "MAE":          mean_absolute_error(y_true, y_pred),
        "MSE":          mse,
        "RMSE":         np.sqrt(mse),
        "MAPE_%":       mean_absolute_percentage_error(y_true, y_pred) * 100,
        "MedianAE":     median_absolute_error(y_true, y_pred),
        "MaxError":     max_error(y_true, y_pred),
        "ExplainedVar": explained_variance_score(y_true, y_pred),
        "RMSLE":        rmsle,
    }

## 5. Train and evaluate

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows, preds = [], {}

for name, model in models.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    fit_time = time.time() - t0

    y_pred = model.predict(X_test)
    preds[name] = y_pred
    cv = cross_val_score(model, X_train, y_train, cv=kf, scoring="r2", n_jobs=-1)

    rows.append({"Model": name,
                 "Train_R2": model.score(X_train, y_train),
                 **all_metrics(y_test, y_pred, X_test.shape[1]),
                 "CV_R2_mean": cv.mean(), "CV_R2_std": cv.std(),
                 "Fit_time_s": fit_time})

results = pd.DataFrame(rows).set_index("Model").sort_values("R2", ascending=False)
results["Overfit_gap"] = results["Train_R2"] - results["R2"]

higher_better = ["Train_R2", "R2", "Adj_R2", "ExplainedVar", "CV_R2_mean"]
lower_better  = ["MAE", "MSE", "RMSE", "MAPE_%", "MedianAE", "MaxError", "RMSLE",
                 "CV_R2_std", "Fit_time_s", "Overfit_gap"]
(results.style
    .highlight_max(subset=higher_better, color="#b7e4c7")
    .highlight_min(subset=lower_better, color="#b7e4c7")
    .format("{:.4f}"))

## 6. Visual comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, (metric, title) in zip(axes.ravel(),
        [("R2", "R² (higher is better)"), ("RMSE", "RMSE (lower is better)"),
         ("MAE", "MAE (lower is better)"), ("MAPE_%", "MAPE % (lower is better)")]):
    results[metric].plot(kind="bar", ax=ax, color=["#4c72b0", "#55a868", "#c44e52"][:len(results)])
    ax.set_title(title); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, p) in zip(axes, preds.items()):
    ax.scatter(y_test, p, alpha=0.7)
    lim = [min(y_test.min(), p.min()), max(y_test.max(), p.max())]
    ax.plot(lim, lim, "r--")
    ax.set_title(f"{name}: Actual vs Predicted")
    ax.set_xlabel("Actual"); ax.set_ylabel("Predicted")
plt.tight_layout(); plt.show()

In [ ]:
# Residual distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, (name, p) in zip(axes, preds.items()):
    sns.histplot(y_test - p, kde=True, ax=ax)
    ax.set_title(f"{name}: residuals")
plt.tight_layout(); plt.show()

## 7. Feature importance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, model) in zip(axes, models.items()):
    pd.Series(model.feature_importances_, index=X.columns).sort_values().plot(kind="barh", ax=ax)
    ax.set_title(f"{name}")
plt.tight_layout(); plt.show()

## 8. Hyperparameter tuning (all three models)

In [ ]:
param_grids = {
    "Decision Tree": {"max_depth": [3, 5, 8, 12, None], "min_samples_split": [2, 5, 10, 20],
                      "min_samples_leaf": [1, 2, 4, 8], "max_features": [None, "sqrt", "log2"]},
    "Random Forest": {"n_estimators": [200, 300, 500], "max_depth": [None, 8, 12, 20],
                      "min_samples_split": [2, 5, 10], "min_samples_leaf": [1, 2, 4],
                      "max_features": ["sqrt", "log2", 1.0]},
    "XGBoost":       {"n_estimators": [200, 300, 500], "learning_rate": [0.01, 0.05, 0.1],
                      "max_depth": [3, 4, 6], "subsample": [0.7, 0.85, 1.0],
                      "colsample_bytree": [0.7, 0.85, 1.0], "min_child_weight": [1, 3, 5]},
}

tuned_rows, tuned_models = [], {}
for name, model in models.items():
    search = RandomizedSearchCV(model, param_grids[name], n_iter=30, cv=kf, scoring="r2",
                                random_state=RANDOM_STATE, n_jobs=-1)
    search.fit(X_train, y_train)
    tuned_models[name] = search.best_estimator_
    y_pred = search.best_estimator_.predict(X_test)
    tuned_rows.append({"Model": f"{name} (tuned)", "CV_R2": search.best_score_,
                       **all_metrics(y_test, y_pred, X_test.shape[1])})
    print(f"{name}: {search.best_params_}")

tuned = pd.DataFrame(tuned_rows).set_index("Model").sort_values("R2", ascending=False)
tuned.round(4)

In [ ]:
# Before vs after tuning
compare = pd.DataFrame({
    "Untuned R2": results["R2"],
    "Tuned R2":   pd.Series({n: tuned.loc[f"{n} (tuned)", "R2"] for n in models}),
    "Untuned RMSE": results["RMSE"],
    "Tuned RMSE":   pd.Series({n: tuned.loc[f"{n} (tuned)", "RMSE"] for n in models}),
})
compare.round(4)

## 9. Save the best model and predict

In [ ]:
candidates = dict(models)
candidates.update({f"{n} (tuned)": m for n, m in tuned_models.items()})
scores = {n: r2_score(y_test, m.predict(X_test)) for n, m in candidates.items()}
champion_name = max(scores, key=scores.get)
champion = candidates[champion_name]
print(f"Champion: {champion_name}  (test R² = {scores[champion_name]:.4f})")

joblib.dump(champion, "car_price_best_model.joblib")
joblib.dump(list(X.columns), "model_columns.joblib")

# Example: 2015 Petrol, Dealer, Manual, 5 lakh showroom price, 40,000 km, first owner
new_car = {"Present_Price": 5.0, "Kms_Driven": 40000, "Owner": 0,
           "Car_Age": pd.Timestamp.today().year - 2015,
           "Fuel_Type_Diesel": 0, "Fuel_Type_Petrol": 1,
           "Seller_Type_Individual": 0, "Transmission_Manual": 1}

model = joblib.load("car_price_best_model.joblib")
cols = joblib.load("model_columns.joblib")
new_df = pd.DataFrame([new_car]).reindex(columns=cols, fill_value=0)
print(f"Predicted resale price: ₹{model.predict(new_df)[0]:.2f} lakhs")